<a href="https://colab.research.google.com/github/LuisPrieto123/MINE_4210_ADL_202520/blob/main/mine__4210_adl_202520_l9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![Logo ADL](https://github.com/nicolastibata/MINE_4210_ADL_202520/blob/main/docs/images/logo.png?raw=true)


# **Laboratorio 9: BERT + Generación de texto**
**Tutor: Nicolás Tibatá**

## **Tabla de Contenido**

[Contexto y objetivos](#scrollTo=5KnQpgpopi8a)<br>
[1. Introducción de los datos](#scrollTo=VjA8zwzJvmeO)<br>
[2. Preparación y Modelamiento](#scrollTo=kG8XHROzvuEH)<br>
[3. Taller 4](#scrollTo=JTKc52_Wvs_N)<br>

### **Contexto y Objetivos**
## Contexto y Objetivos

- Se requiere realizar el análisis de sentimientos de un conjunto de frases del sector financiero como parte de la evaluación del sentimiento del mercado y la reputación de empresas, con el objetivo de tomar decisiones de inversión informadas.

### **Objetivos**
1. Construir una Red Neuronal basada en una arquitectura Transformers para llevar a cabo un análisis de sentimientos, ejemplificando así la aplicación de modelos de procesamiento del lenguaje natural.
2. Generar texto en base a gpt2 de manera de ejemplo.
3. Aplicar de manera combinada ambas herramientas en el taller 4.



**Datos:** [finance-sentence](https://www.kaggle.com/datasets/sbhatti/financial-sentiment-analysis)

### **1. Introducción a los datos**

In [ ]:
!pip install torch -q
!pip install keras-tuner -q
!pip install transformers -q
!pip install "tf-models-official==2.13.*" -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.4/269.4 kB 10.3 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [ ]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
import shutil
import pandas as pd
import numpy as np
import seaborn as sns


import matplotlib.pyplot as plt
%matplotlib inline

import tensorflow as tf
import keras_tuner as kt

from keras.models import Sequential
from keras.layers import Input, Dense, Dropout, LSTM, Flatten
from transformers import TFBertForSequenceClassification, BertTokenizer

from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping
from google.colab import files

import tensorflow_hub as hub
import tensorflow_text

from google.colab import files
from google.colab import userdata

In [ ]:
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')

!kaggle datasets download -d sbhatti/financial-sentiment-analysis
!unzip "financial-sentiment-analysis.zip"

Dataset URL: https://www.kaggle.com/datasets/sbhatti/financial-sentiment-analysis
License(s): CC0-1.0
financial-sentiment-analysis.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  financial-sentiment-analysis.zip
replace data.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: data.csv                


In [ ]:
# Devidir los datos en train y test
data = pd.read_csv('/content/data.csv', sep=',')
data

,Sentence,Sentiment
0,The GeoSolutions technology will leverage Bene...,positive
1,"$ESI on lows, down $1.50 to $2.50 BK a real po...",negative
2,"For the last quarter of 2010 , Componenta 's n...",positive
3,According to the Finnish-Russian Chamber of Co...,neutral
4,The Swedish buyout firm has sold its remaining...,neutral
...,...,...
5837,RISING costs have forced packaging producer Hu...,negative
5838,Nordic Walking was first used as a summer trai...,neutral
5839,"According shipping company Viking Line , the E...",neutral
5840,"In the building and home improvement trade , s...",neutral


Veamos la distribución de los sentimientos en el dataset

In [ ]:
data["Sentiment"].value_counts()

,count
Sentiment,
neutral,3130
positive,1852
negative,860


### **2. Preparación y Modelamiento**

#### **Preparación**

Pasamos el sentimiento (clase) a un valor entero usando LabelEncoder

In [ ]:
label_encoder = LabelEncoder()
data['Sentiment'] = label_encoder.fit_transform(data['Sentiment'])

etiquetas_unicas = label_encoder.classes_
for valor_numerico, etiqueta_original in enumerate(etiquetas_unicas):
    print(f'Valor numérico: {valor_numerico}, Etiqueta original: {etiqueta_original}')

Valor numérico: 0, Etiqueta original: negative
Valor numérico: 1, Etiqueta original: neutral
Valor numérico: 2, Etiqueta original: positive


In [ ]:
# Divide los datos en entrenamiento y prueba
train, test = train_test_split(data, test_size=0.2, stratify=data['Sentiment'], random_state=42, shuffle = True)

# Ahora divide el conjunto de entrenamiento en entrenamiento y validación
train, val = train_test_split(train, test_size=0.2, stratify=train['Sentiment'], random_state=42, shuffle = True)

print("Tamaño de datos de entrenamiento:", train.shape)
print("Tamaño de datos de validación:", val.shape)
print("Tamaño de datos de prueba:", test.shape)

train

Tamaño de datos de entrenamiento: (3738, 2)
Tamaño de datos de validación: (935, 2)
Tamaño de datos de prueba: (1169, 2)


,Sentence,Sentiment
3097,"Digia will also set up two subsidiaries , Digi...",1
5048,$BBRY Sierra. Has a great cash balance and imp...,2
5727,"Britain's FTSE gains, Land Securities up after...",2
185,The Finnish company sold its UK operation - co...,1
4265,Russian Media Ventures ' minority shareholder ...,1
...,...,...
326,( I&H ) in a move to enhance growth .,2
2821,"In addition , a further 29 employees can be la...",0
4365,"The paper industry 's de-inking sludge , which...",1
1603,$JE LOOKS like we are bouncing. Would be nice...,2


In [ ]:
X_train, X_test, X_val= train['Sentence'], test['Sentence'], val['Sentence']
y_train, y_test, y_val= train['Sentiment'], test['Sentiment'], val['Sentiment']

print("x_train", X_train.shape, " y_train:", y_train.shape)
print("x_test:", X_test.shape, "y_test", y_test.shape)
print("x_val", X_val.shape, "y_val:", y_val.shape)

x_train (3738,)  y_train: (3738,)
x_test: (1169,) y_test (1169,)
x_val (935,) y_val: (935,)


#### **Modelamiento BERT**

 Con el código de las siguientes secciones podemos seleccionar uno de los modelos BERT y su correspondiente modelo de preprocesamiento utilizando TensorFlow Hub. Pueden ver más información [aquí](https://huggingface.co/transformers/v3.0.2/model_doc/bert.html).

In [ ]:
# Cargar los modelos de TensorFlow Hub con trainable=True si deseas fine-tuning
bert_preprocess_model = hub.KerasLayer(
    "https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3",
    name='preprocessing'
)

bert_model = hub.KerasLayer(
    "https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-4_H-512_A-8/2",
    trainable=False, # Solo como extractor de características
    name='BERT_encoder'
)

print(f'BERT model selected: {bert_model}')
print(f'Preprocess model selected: {bert_preprocess_model}')

BERT model selected: <tensorflow_hub.keras_layer.KerasLayer object at 0x7911531dc1a0>
Preprocess model selected: <tensorflow_hub.keras_layer.KerasLayer object at 0x7912447c8320>


In [ ]:
# Capas Bert
text_input = tf.keras.layers.Input(shape=(), dtype=tf.string, name='text')
preprocessing_output = bert_preprocess_model(text_input)
outputs = bert_model(preprocessing_output)

# Capas de nuestra red adicional
#l = tf.keras.layers.Dropout(0.1, name="dropout")
l = tf.keras.layers.Dense(512, activation='sigmoid', name="capaOculta1")(outputs['pooled_output'])
l = tf.keras.layers.Dropout(0.2, name="dropout")(l)
y = tf.keras.layers.Dense(3, activation='softmax', name="output")(l)

# Usamos inputs y outputs para construir el modelo final
model = tf.keras.Model(inputs=[text_input], outputs = [y])

In [ ]:
metrics = ['accuracy']

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=metrics)

In [ ]:
print(model.summary())

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 text (InputLayer)           [(None,)]                    0         []                            
                                                                                                  
 preprocessing (KerasLayer)  {'input_type_ids': (None,    0         ['text[0][0]']                
                             128),                                                                
                              'input_mask': (None, 128)                                           
                             , 'input_word_ids': (None,                                           
                              128)}                                                               
                                                                                              

In [ ]:
model.fit(X_train, y_train,
          validation_data = (X_val, y_val),
          epochs=20,
          #callbacks= EarlyStopping(monitor='val_accuracy', patience=4)
          )

Epoch 1/20
117/117 [==============================] - 19s 94ms/step - loss: 0.8734 - accuracy: 0.5998 - val_loss: 0.7675 - val_accuracy: 0.6374
Epoch 2/20
117/117 [==============================] - 10s 89ms/step - loss: 0.7824 - accuracy: 0.6359 - val_loss: 0.7596 - val_accuracy: 0.6160
Epoch 3/20
117/117 [==============================] - 10s 89ms/step - loss: 0.7396 - accuracy: 0.6522 - val_loss: 0.7048 - val_accuracy: 0.6471
Epoch 4/20
117/117 [==============================] - 13s 110ms/step - loss: 0.7286 - accuracy: 0.6605 - val_loss: 0.7377 - val_accuracy: 0.6299
Epoch 5/20
117/117 [==============================] - 10s 90ms/step - loss: 0.7066 - accuracy: 0.6747 - val_loss: 0.6869 - val_accuracy: 0.6652
Epoch 6/20
117/117 [==============================] - 14s 120ms/step - loss: 0.6955 - accuracy: 0.6734 - val_loss: 0.6954 - val_accuracy: 0.6663
Epoch 7/20
117/117 [==============================] - 16s 134ms/step - loss: 0.6814 - accuracy: 0.6814 - val_loss: 0.6715 - val_accura

**Evaluamos nuestro modelo**

In [ ]:
y_pred = model.predict(X_train)
y_pred = np.argmax(y_pred, axis=1)

print(classification_report(y_train, y_pred))

In [ ]:
y_pred = model.predict(X_test)
y_pred = np.argmax(y_pred, axis=1)

print(classification_report(y_test, y_pred))

#### **Generación de Texto**

Utilizamos un modelo GPT transformer para generación de texto, el proceso es similar al uso de BERT

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

In [ ]:
model_name = "gpt2"
model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

In [ ]:
input_text = "Once upon a time"
input_ids = tokenizer.encode(input_text, return_tensors="pt")

In [ ]:
output = model.generate(
    input_ids,
    max_length=50,           # Longitud máxima del texto generado
    num_return_sequences=1,  # Cantidad de sentencias generadas
    temperature=0.7,         # Temperatura (más alto = más creativo)
    top_k=50,                # Top 50 de palabras probables
    top_p=0.9,               # Centrado a palabras de alta probabilidad
    repetition_penalty=1.2,  # Reducción de repetición de palabras
    do_sample=True           # Habilita el randomness del texto generado
)

In [ ]:
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(generated_text)

### **3. Taller 4**


Instrucciones

1. El archivo a presentar debe ser en formato .ipynb o HTML con sus celdas ejecutadas. Celdas sin ejecutar no podrán ser evaluadas.
2. El nombre del archivo debe ser taller_4_{Apellido_Nombre}_{Apellido_Nombre} de cada integrante del equipo.
3. Las entregas solo se hacen a través de Bloque Neón.

------


1. Los resultados del modelo base no son los mejores. Utilice el modelo [Distilbert](https://huggingface.co/transformers/v3.0.2/model_doc/distilbert.html) de Huggingface aplicando padding a los diferentes dataframes (train, val, test). ¿Qué es Distilbert? ¿Mejoran los resultados? ¿El entrenamiento es más rápido? Compare los resultados con el modelo base del laboratorio.
```python
# clue

import tensorflow as tf
from transformers import DistilBertTokenizer, TFDistilBertForSequenceClassification
from sklearn.model_selection import train_test_split
# Tokenización
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
X_train = [tokenizer.encode(text, add_special_tokens=True, truncation=True, max_length=128) for text in X_train]
X_val = [tokenizer.encode(text, add_special_tokens=True, truncation=True, max_length=128) for text in X_val]
X_test = [tokenizer.encode(text, add_special_tokens=True, truncation=True, max_length=128) for text in X_test]

# Padding
X_train = tf.keras.preprocessing.sequence.pad_sequences(...)
X_val = tf.keras.preprocessing.sequence.pad_sequences(...)
X_test = tf.keras.preprocessing.sequence.pad_sequences(...)

model = TFDistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased")
```

2. Genere sentencias nuevas con el modelo gpt2 o algún otro de preferencia para poder tener datos sintéticos y aumentar la cantidad de sentencias negativas del dataset original (aumentar las sentencias negativas hasta 1200 datos de esa clase). Luego reentrene el modelo del punto 1 (es decir el modelo entrenado con Distilbert) con este nuevo dataset aumentado y documente las métricas de evaluación.


3. Realice fine-tuning al modelo base del laboratorio y compárelo con el modelo del punto 2. ¿Que diferencias nota al realizar este paso adicional?
``` python
# clue

trainable=True,

```


**Desarrollo del Taller**

**Punto 1**

A continuación se incluyen las celdas con el código necesario para utilizar Distilbert, se hace una primera configuración para utilizar DISTILBERT con TensorFlow sin embargo, no fué posible la ejecución del modelo por incompatibilidad de clases, la plataforma recomienda la utilización de Clases PyTorch.


In [ ]:
# clue

import tensorflow as tf
from transformers import DistilBertTokenizer, TFDistilBertForSequenceClassification
from sklearn.model_selection import train_test_split

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

# Reemplaza todo tu código de tokenización por esto:
X_train = [tokenizer.encode(text, add_special_tokens=True, truncation=True, max_length=128) for text in X_train]
X_val = [tokenizer.encode(text, add_special_tokens=True, truncation=True, max_length=128) for text in X_val]
X_test = [tokenizer.encode(text, add_special_tokens=True, truncation=True, max_length=128) for text in X_test]

from tensorflow.keras.preprocessing.sequence import pad_sequences

# Aplicar padding después de tokenizar
X_train = pad_sequences(X_train, maxlen=128, dtype='int32', padding='post', truncating='post')
X_val = pad_sequences(X_val, maxlen=128, dtype='int32', padding='post', truncating='post')
X_test = pad_sequences(X_test, maxlen=128, dtype='int32', padding='post', truncating='post')

model = TFDistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=3, local_files_only=False)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


TypeError: 'builtins.safe_open' object is not iterable

Uso del modelo Distilbert de Huggingface con Pytorch y agregando la capa de personalización equivalente utilizada previamente con BERT

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm import tqdm
import numpy as np

# 1. Verificar si hay GPU disponible
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando dispositivo: {device}')

# 2. Cargar el tokenizer
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# 3. Tokenizar y aplicar padding a los dataframes
X_train = tokenizer(
    train['Sentence'].tolist(),
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'  # PyTorch tensors
)

X_val = tokenizer(
    val['Sentence'].tolist(),
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

X_test = tokenizer(
    test['Sentence'].tolist(),
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

# 4. Crear clase Dataset personalizada
class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels.iloc[idx])
        return item

# 5. Crear datasets
train_dataset = SentimentDataset(X_train, train['Sentiment'])
val_dataset = SentimentDataset(X_val, val['Sentiment'])
test_dataset = SentimentDataset(X_test, test['Sentiment'])

# 6. Crear DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# 7. Crear modelo personalizado con capas adicionales
class DistilBertWithCustomLayers(torch.nn.Module):
    def __init__(self, num_labels=3):
        super(DistilBertWithCustomLayers, self).__init__()

        # Cargar DistilBERT base
        self.distilbert = DistilBertForSequenceClassification.from_pretrained(
            'distilbert-base-uncased',
            num_labels=num_labels
        ).distilbert  # Solo el encoder, sin la capa de clasificación

        # Capas personalizadas
        self.dense_hidden = torch.nn.Linear(768, 512)  # 768 es la dim de DistilBERT
        self.sigmoid = torch.nn.Sigmoid()
        self.dropout = torch.nn.Dropout(0.2)
        self.classifier = torch.nn.Linear(512, num_labels)

    def forward(self, input_ids, attention_mask):
        # Pasar por DistilBERT
        outputs = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Obtener el embedding del token [CLS] (primer token)
        hidden_state = outputs.last_hidden_state[:, 0, :]  # [batch_size, 768]

        # Pasar por capas personalizadas
        x = self.dense_hidden(hidden_state)  # [batch_size, 512]
        x = self.sigmoid(x)
        x = self.dropout(x)
        logits = self.classifier(x)  # [batch_size, num_labels]

        return logits

# Instanciar el modelo
model = DistilBertWithCustomLayers(num_labels=3)

# ============ CONGELAR DISTILBERT ============
for param in model.distilbert.parameters():
    param.requires_grad = False
# ============================================

# Contar parámetros
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parámetros: {total_params:,}")
print(f"Parámetros entrenables: {trainable_params:,}")
print(f"Parámetros congelados: {total_params - trainable_params:,}")

model.to(device)

# ==================== VISUALIZACIÓN DE ARQUITECTURA ====================
print("\n" + "="*80)
print("ARQUITECTURA DEL MODELO - DistilBERT + Capas Personalizadas")
print("="*80)
print(model)
print("-"*80)
print(f"\n{'RESUMEN':^80}")
print("-"*80)
print(f"Total de parámetros:      {total_params:>20,}")
print(f"Parámetros entrenables:   {trainable_params:>20,}")
print(f"Parámetros congelados:    {total_params - trainable_params:>20,}")
print(f"Porcentaje entrenable:    {100 * trainable_params / total_params:>19.2f}%")
print("="*80 + "\n")
# ========================================================================


# 8. Configurar optimizador y scheduler
optimizer = AdamW(model.parameters(), lr=5e-5)
num_epochs = 20
total_steps = len(train_loader) * num_epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# 9. Función de entrenamiento (MODIFICADA)
def train_epoch(model, data_loader, optimizer, scheduler, device):
    model.train()
    losses = []
    correct_predictions = 0
    loss_fn = torch.nn.CrossEntropyLoss()  # Función de pérdida

    for batch in tqdm(data_loader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = loss_fn(logits, labels)

        # Calcular accuracy
        _, preds = torch.max(logits, dim=1)
        correct_predictions += torch.sum(preds == labels)

        losses.append(loss.item())

        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    return correct_predictions.double() / len(data_loader.dataset), np.mean(losses)


# 10. Función de evaluación (MODIFICADA)
def eval_model(model, data_loader, device):
    model.eval()
    losses = []
    correct_predictions = 0
    loss_fn = torch.nn.CrossEntropyLoss()

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = loss_fn(logits, labels)

            _, preds = torch.max(logits, dim=1)
            correct_predictions += torch.sum(preds == labels)
            losses.append(loss.item())

    return correct_predictions.double() / len(data_loader.dataset), np.mean(losses)

# 11. Entrenamiento
history = {'train_acc': [], 'train_loss': [], 'val_acc': [], 'val_loss': []}

for epoch in range(num_epochs):
    print(f'\nEpoch {epoch + 1}/{num_epochs}')
    print('-' * 50)

    train_acc, train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    print(f'Train loss: {train_loss:.4f}, Train accuracy: {train_acc:.4f}')

    val_acc, val_loss = eval_model(model, val_loader, device)
    print(f'Val loss: {val_loss:.4f}, Val accuracy: {val_acc:.4f}')

    history['train_acc'].append(train_acc)
    history['train_loss'].append(train_loss)
    history['val_acc'].append(val_acc)
    history['val_loss'].append(val_loss)

# 12. Evaluación final en test
test_acc, test_loss = eval_model(model, test_loader, device)
print(f'\nTest loss: {test_loss:.4f}, Test accuracy: {test_acc:.4f}')


Usando dispositivo: cuda


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Total parámetros: 66,758,147
Parámetros entrenables: 395,267
Parámetros congelados: 66,362,880

Epoch 1/20
--------------------------------------------------


Training: 100%|██████████| 59/59 [00:11<00:00,  5.24it/s]


Train loss: 0.9838, Train accuracy: 0.5326


...

Train loss: 0.8088, Train accuracy: 0.6268


Evaluating: 100%|██████████| 15/15 [00:03<00:00,  4.83it/s]


Val loss: 0.7974, Val accuracy: 0.6171


Evaluating: 100%|██████████| 19/19 [00:03<00:00,  4.86it/s]


Test loss: 0.7904, Test accuracy: 0.6510


Punto 2

Se agregan celdas de código para utilizar Generador GPT2 y generar de manera sintética 340 sentencias negativas que se agregaran al dataset inicial para incrementar la cantidad de datos de la clase de Sentimiento negativo a 1200

In [ ]:
from transformers import pipeline
import pandas as pd
import numpy as np

print("🔄 Cargando modelo GPT-2...")
# Crear pipeline de generación de texto (usa PyTorch en el backend)
generator = pipeline('text-generation', model='gpt2', framework='pt')

# Prompts para generar sentencias negativas
negative_prompts = [
    "I hate",
    "I'm disappointed",
    "This is terrible",
    "I feel sad",
    "The worst thing",
    "I'm angry about",
    "I dislike",
    "This makes me upset",
    "I regret",
    "I'm frustrated by",
    "It's awful",
    "I can't stand",
    "This is horrible",
    "I feel terrible",
    "What a disaster",
    "I'm unhappy with",
    "This is disappointing",
    "I'm annoyed by",
    "I despise",
    "This ruins",
    "I'm sick of",
    "This bothers me",
    "I'm worried about",
    "This is unacceptable",
    "I feel miserable",
    "This is the worst",
    "I'm depressed about",
    "This makes me furious",
    "I'm disgusted by",
    "This is so bad",
    "I never want to",
    "This hurts",
    "I'm upset that",
    "This is painful",
]

sentences = []
num_sentences = 340
sentences_per_prompt = num_sentences // len(negative_prompts) + 1

print(f"\n📝 Generando {num_sentences} sentencias con sentimiento negativo...\n")

for i, prompt in enumerate(negative_prompts):
    if len(sentences) >= num_sentences:
        break

    # Calcular cuántas sentencias generar con este prompt
    needed = min(sentences_per_prompt, num_sentences - len(sentences))

    for j in range(needed):
        try:
            # Generar texto
            result = generator(
                prompt,
                max_new_tokens=20,       # Longitud máxima de nuevos tokens generados
                num_return_sequences=1,  # Una sentencia a la vez
                temperature=0.9,         # Creatividad
                top_k=50,
                top_p=0.95,
                do_sample=True,
                pad_token_id=50256,      # Token de padding de GPT-2
                truncation=True
            )

            # Extraer y limpiar la sentencia
            sentence = result[0]['generated_text'].strip()

            # Tomar solo la primera oración (hasta el primer punto)
            if '.' in sentence:
                sentence = sentence.split('.')[0] + '.'
            elif '!' in sentence:
                sentence = sentence.split('!')[0] + '!'
            elif '?' in sentence:
                sentence = sentence.split('?')[0] + '?'
            else:
                sentence = sentence + '.'

            # Limpiar saltos de línea y espacios múltiples
            sentence = ' '.join(sentence.split())

            sentences.append(sentence)

            # Mostrar progreso cada 50 sentencias
            if len(sentences) % 50 == 0:
                print(f"✓ Generadas {len(sentences)}/{num_sentences} sentencias...")

        except Exception as e:
            print(f"⚠️ Error generando sentencia: {e}")
            continue

# Asegurar que tenemos exactamente 340 sentencias
sentences = sentences[:num_sentences]

print(f"\n✅ Se generaron {len(sentences)} sentencias con éxito!\n")

# Crear DataFrame
df = pd.DataFrame({
    'Sentence': sentences,
    'Sentiment': ['negative'] * len(sentences)  # Todas etiquetadas como negativas
})

# Mostrar primeras 10 sentencias
print("="*80)
print("PRIMERAS 10 SENTENCIAS GENERADAS:")
print("="*80)
for idx, row in df.head(10).iterrows():
    print(f"{idx+1}. {row['Sentence']}")
    print(f"   Sentimiento: {row['Sentiment']}\n")

# Mostrar estadísticas
print("="*80)
print("ESTADÍSTICAS:")
print("="*80)
print(f"Total de sentencias:        {len(df)}")
print(f"Sentencias negativas:       {(df['Sentiment'] == 'negative').sum()}")
print(f"Longitud promedio:          {df['Sentence'].str.len().mean():.1f} caracteres")
print(f"Longitud mínima:            {df['Sentence'].str.len().min()} caracteres")
print(f"Longitud máxima:            {df['Sentence'].str.len().max()} caracteres")
print("="*80)

# Guardar en CSV
csv_filename = 'negative_sentences_340.csv'
df.to_csv(csv_filename, index=False, encoding='utf-8')
print(f"\n💾 Archivo guardado: '{csv_filename}'")

# Guardar también en formato texto plano
txt_filename = 'negative_sentences_340.txt'
with open(txt_filename, 'w', encoding='utf-8') as f:
    for idx, row in df.iterrows():
        f.write(f"{idx+1}. {row['Sentence']} | Sentimiento: {row['Sentiment']}\n")
print(f"💾 Archivo guardado: '{txt_filename}'")

# Mostrar algunas sentencias aleatorias
print("\n" + "="*80)
print("10 SENTENCIAS ALEATORIAS:")
print("="*80)
random_samples = df.sample(n=10, random_state=42)
for idx, row in random_samples.iterrows():
    print(f"• {row['Sentence']}")

print("\n✨ ¡Proceso completado exitosamente!")

🔄 Cargando modelo GPT-2...


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

✓ Generadas 250/340 sentencias...


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

✓ Generadas 300/340 sentencias...


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both


✅ Se generaron 340 sentencias con éxito!

PRIMERAS 10 SENTENCIAS GENERADAS:
1. I hate to say it, but the other thing is, the fact is that you're just so wrong, and we've got so many things wrong with this.
   Sentimiento: negative

2. I hate that it's happening to him, it's not.
   Sentimiento: negative

3. I hate myself for being an asshole.
   Sentimiento: negative

4. I hate being called a 'puppy' but I hate being told I'm not a 'puppy.
   Sentimiento: negative

5. I hate to say it, but I know they're not happy about it.
   Sentimiento: negative

6. I hate the way they're going, and they're going to do it all the time," said D.
   Sentimiento: negative

7. I hate when people have to tell you they are stupid because you don't understand, so I know how annoying you get.
   Sentimiento: negative

8. I hate her.
   Sentimiento: negative

9. I hate to say it out loud, but my love for the original Tic Tac isn't the only reason I still have my tic tac and I still love the colors.
   Senti